In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 16 consolidation — Attention moves information; training gives it task meaning

Read `sessions/16_session.md` first. This is an executed reference, not an assessment of you.

**Evidence:** Real stereo-image patches processed by an explicitly untrained attention operator; no VGGT predictions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT/'data/raw/iris.csv').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Start Jupyter in the extracted listening_to_shape_lab folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))


## Step 1

Extract actual image patches; these are raw flattened pixels, not trained embeddings.

Before running: write your expected result and name one way this check could fail.

In [3]:
from shape_lab.stereo import reference_sample
from shape_lab.attention import patch_tokens, attention
left, _, _, _ = reference_sample()
tokens, rc = patch_tokens(left[:32,:32], patch_size=8)
tokens = tokens/255.
print('patch tokens:', tokens.shape, 'locations:', rc.shape)

patch tokens: (16, 192) locations: (16, 2)


## Step 2

A zero query must average the values. Check this directly.

Before running: write your expected result and name one way this check could fail.

In [4]:
query = np.zeros((1,tokens.shape[1]))
output, weights = attention(query, tokens, tokens)
assert np.allclose(weights, 1/len(tokens))
assert np.allclose(output[0], tokens.mean(axis=0))
print('attention weight sum:', weights.sum())

attention weight sum: 1.0


## Step 3

For arbitrary allowed weights, constant values must remain constant.

Before running: write your expected result and name one way this check could fail.

In [5]:
Q = tokens[:3]; K = tokens
V = np.tile([2., -1., .5], (len(K),1))
out, w = attention(Q, K, V)
assert np.allclose(w.sum(axis=1), 1.)
assert np.allclose(out, [2., -1., .5])
result = {'real_image_patch_count': len(tokens), 'zero_query_mean_check': 'passed',
          'constant_value_check': 'passed', 'trained_embeddings_used': False,
          'VGGT_inference_executed': False}

## Keep execution evidence separate from learning evidence

The following saves this reference calculation. Write your own explanation in `my_work/`; no learner score is assigned.

In [6]:
result['stage'] = 16
result['execution_role'] = 'reference_consolidation'
result['data_role'] = 'Real stereo-image patches processed by an explicitly untrained attention operator; no VGGT predictions.'
output = ROOT/'reports/consolidation_v4/16.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Saved reference result:', output.relative_to(ROOT))


Saved reference result: reports/consolidation_v4/16.json
